# 🧠 Parte 1 — Do texto aos números: tokens e embeddings

**Aula: Como funciona um Transformer na prática (mecanismo de atenção)**

Nesta primeira parte vamos responder a uma pergunta simples:

> **Como um computador, que só entende números, consegue "ler" uma frase?**

Vamos fazer tudo com Python puro: listas, dicionários e laços `for`. Nada de bibliotecas mágicas!

📌 **Como usar este notebook:** execute as células **na ordem**, de cima para baixo, com `Shift + Enter`.

## Passo 1 — Quebrando a frase em pedaços (tokens)

O primeiro passo de qualquer modelo de linguagem é **dividir o texto em pedaços menores**, chamados de **tokens**.

Nos modelos reais, um token pode ser um pedaço de palavra (ex.: `"compu"` + `"tador"`). Aqui, para simplificar, **cada palavra será um token**.

In [ ]:
frase = "o gato bebeu leite"

# split(" ") corta o texto em todos os espaços
palavras = frase.split(" ")

print("Frase original:", frase)
print("Tokens:", palavras)
print("Quantidade de tokens:", len(palavras))

## Passo 2 — Criando um vocabulário

O computador não guarda palavras, guarda **números**. Então vamos dar um **número de identificação (ID)** para cada palavra diferente que aparecer nos nossos textos.

Esse "dicionário de palavras" se chama **vocabulário**.

In [ ]:
textos = [
    "o gato bebeu leite",
    "o cachorro comeu queijo",
    "o gato comeu queijo",
]

vocabulario = {}   # dicionário vazio
proximo_id = 0

for texto in textos:
    for palavra in texto.split(" "):
        if palavra not in vocabulario:          # palavra nova?
            vocabulario[palavra] = proximo_id   # ganha um ID
            proximo_id = proximo_id + 1

print("Vocabulário:")
for palavra in vocabulario:
    print("  ", palavra, "->", vocabulario[palavra])

In [ ]:
# Transformando a nossa frase em uma lista de IDs
ids = []
for palavra in palavras:
    ids.append(vocabulario[palavra])

print("Tokens:", palavras)
print("IDs:   ", ids)

## 🤔 Problema: o ID não tem significado!

Repare: `gato` tem um ID e `cachorro` tem outro. Mas o número **não diz nada** sobre a palavra.
Pelo ID, o computador não sabe que *gato* e *cachorro* são parecidos (os dois são animais), nem que *leite* e *queijo* são comida.

**Solução:** representar cada palavra como uma **lista de números com significado**. Essa lista se chama **vetor de embedding**.

Vamos inventar 3 "características" para nossas palavras:

| Posição do vetor | Pergunta que ela responde |
|---|---|
| 0 | É um **animal**? |
| 1 | É **comida/bebida**? |
| 2 | É uma **ação**? |

Cada número vai de 0 (nada) até 1 (muito).

In [ ]:
#                [animal, comida, ação]
embeddings = {
    "o":         [0.1,    0.1,    0.1],
    "gato":      [0.9,    0.1,    0.0],
    "cachorro":  [0.8,    0.1,    0.1],
    "leite":     [0.0,    0.9,    0.1],
    "queijo":    [0.1,    0.8,    0.0],
    "bebeu":     [0.1,    0.3,    0.9],
    "comeu":     [0.1,    0.4,    0.8],
}

for palavra in embeddings:
    print(palavra, "->", embeddings[palavra])

> 💡 **Importante:** aqui **nós** escolhemos os números à mão. Em um Transformer de verdade, os vetores têm centenas ou milhares de posições (ex.: 768) e os números são **aprendidos sozinhos** durante o treinamento, lendo bilhões de textos.

Agora, a frase deixa de ser texto e vira uma **lista de vetores**:

In [ ]:
print("Frase:", frase)
print()
for palavra in palavras:
    print(palavra, "->", embeddings[palavra])

## Passo 3 — Medindo se duas palavras são parecidas: o **produto escalar**

Para saber se dois vetores "combinam", fazemos uma conta muito simples:

1. Multiplicamos posição por posição;
2. Somamos tudo.

Exemplo, à mão, entre **gato** `[0.9, 0.1, 0.0]` e **cachorro** `[0.8, 0.1, 0.1]`:

```
0.9 × 0.8 = 0.72
0.1 × 0.1 = 0.01
0.0 × 0.1 = 0.00
            ----
soma      = 0.73   ← número alto = parecidos!
```

Vamos fazer o computador fazer essa conta mostrando cada passo:

In [ ]:
gato = embeddings["gato"]
cachorro = embeddings["cachorro"]

soma = 0
for i in range(3):
    multiplicacao = gato[i] * cachorro[i]
    print("posição", i, ":", gato[i], "x", cachorro[i], "=", round(multiplicacao, 2))
    soma = soma + multiplicacao

print("Produto escalar (gato, cachorro) =", round(soma, 2))

Como vamos usar essa conta **muitas vezes**, vamos guardá-la em uma função simples:

In [ ]:
def produto_escalar(vetor_a, vetor_b):
    soma = 0
    for i in range(len(vetor_a)):
        soma = soma + vetor_a[i] * vetor_b[i]
    return round(soma, 2)

print("gato  x cachorro =", produto_escalar(embeddings["gato"], embeddings["cachorro"]))
print("gato  x leite    =", produto_escalar(embeddings["gato"], embeddings["leite"]))
print("leite x queijo   =", produto_escalar(embeddings["leite"], embeddings["queijo"]))
print("bebeu x comeu    =", produto_escalar(embeddings["bebeu"], embeddings["comeu"]))

## Passo 4 — Comparando uma palavra com todas as outras

Vamos ver com quais palavras o **gato** mais "combina":

In [ ]:
escolhida = "gato"

print("Comparando", escolhida, "com todas as palavras:")
for palavra in embeddings:
    valor = produto_escalar(embeddings[escolhida], embeddings[palavra])
    print("  ", escolhida, "x", palavra, "=", valor)

✅ Palavras do **mesmo tipo** dão números **maiores**. É exatamente essa ideia de "medir o quanto uma palavra combina com outra" que o **mecanismo de atenção** vai usar nas próximas partes!

---
## ✏️ Exercício

1. Adicione ao dicionário `embeddings` as palavras **"rato"** e **"suco"**, escolhendo números que façam sentido.
2. Calcule o produto escalar de `rato` com `gato` e de `suco` com `leite`.
3. Troque `escolhida = "gato"` por `"bebeu"` e execute de novo. Quem combina mais com "bebeu"?

In [ ]:
# Complete aqui o exercício
embeddings["rato"] = [0.0, 0.0, 0.0]   # troque pelos seus números
embeddings["suco"] = [0.0, 0.0, 0.0]   # troque pelos seus números

print("rato x gato  =", produto_escalar(embeddings["rato"], embeddings["gato"]))
print("suco x leite =", produto_escalar(embeddings["suco"], embeddings["leite"]))

---
## 📝 Resumo da Parte 1

| Conceito | O que é |
|---|---|
| **Token** | Pedaço do texto (aqui, uma palavra) |
| **Vocabulário** | Lista de todos os tokens conhecidos, cada um com um ID |
| **Embedding** | Vetor (lista de números) que representa o *significado* do token |
| **Produto escalar** | Conta que mede o quanto dois vetores "combinam" |

➡️ **Na Parte 2** vamos transformar esses números em **porcentagens de atenção**.